# Project 01 · Digital lending credit risk in SQL

**Benson M. Merita · MeritaAnalytics** · Tools: SQL (SQLite), Python (pandas), Matplotlib

**Business question.** How risky is the loan book, which segments drive defaults, what loss should we expect on active loans, and which approval rule would have made the most money?

**Data.** A lender database with four tables: `customers` (1,000), `loans` (560), `repayments` (2,867) and `transactions`. The data is synthetic (generated with a fixed seed), so it describes no real lender or person. Snapshot date: 31 December 2025.

**How to read this notebook.** Each step starts with a short note: *what* the step does and *why* it matters. The SQL is written out in full so it can be copied into any SQL tool. Python only runs the query and draws the charts.

| Step | What it answers |
|---|---|
| 1 | What is in the database? |
| 2 | Can we trust the data? (7 checks) |
| 3 | Which loans are old enough to judge? |
| 4 | Which segments default most? |
| 5 | Is risk getting better or worse by month? (vintages) |
| 6 | How much of the book is overdue? (PAR30 / PAR90) |
| 7 | What loss should we expect on active loans? |
| 8 | Which approval rule makes the most money? |

## Setup

**What:** load the libraries and connect to the database file.
**Why:** every query below runs against the same file, `lender.db`, which is in the `datasets` folder of this repository.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, DARK, AMBER = "#1F5FD6", "#0B2A4A", "#B45309"
plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)

DB = "../datasets/01-lending-credit-risk/sql/lender.db"
con = sqlite3.connect(DB)

def sql(query):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql(query, con)

## Step 1 · What is in the database?

**What:** count the rows in each table and look at three loans.
**Why:** before any analysis, confirm the data is the size we expect and see what one record looks like.

In [ ]:
sql("""
SELECT 'customers'    AS table_name, COUNT(*) AS rows FROM customers
UNION ALL SELECT 'loans',        COUNT(*) FROM loans
UNION ALL SELECT 'repayments',   COUNT(*) FROM repayments
UNION ALL SELECT 'transactions', COUNT(*) FROM transactions
""")

In [ ]:
sql("""
SELECT loan_id, product, principal_ksh, interest_rate_pct, term_months,
       disbursement_date, credit_score, status, days_past_due
FROM loans
LIMIT 3
""")

## Step 2 · Can we trust the data?

**What:** seven data quality checks in one query. Each line counts the rows that *fail* a rule, so every number should be 0.
**Why:** a risk report built on duplicates, impossible scores or loans with no customer would be wrong before it starts.

In [ ]:
sql("""
SELECT 'duplicate loan_id' AS check_name, COUNT(*) - COUNT(DISTINCT loan_id) AS rows_failing FROM loans
UNION ALL SELECT 'principal <= 0',               COUNT(*) FROM loans WHERE principal_ksh <= 0
UNION ALL SELECT 'credit score outside 300-850', COUNT(*) FROM loans WHERE credit_score NOT BETWEEN 300 AND 850
UNION ALL SELECT 'term_months <= 0',             COUNT(*) FROM loans WHERE term_months <= 0
UNION ALL SELECT 'loan with unknown customer',   COUNT(*) FROM loans
          WHERE customer_id NOT IN (SELECT customer_id FROM customers)
UNION ALL SELECT 'repayment with unknown loan',  COUNT(*) FROM repayments
          WHERE loan_id NOT IN (SELECT loan_id FROM loans)
UNION ALL SELECT 'defaulted but days_past_due <= 90', COUNT(*) FROM loans
          WHERE status = 'Defaulted' AND days_past_due <= 90
""")

All seven checks return 0, so the data can be used. The status values also agree with days past due:

In [ ]:
sql("""
SELECT status, COUNT(*) AS loans, MIN(days_past_due) AS min_dpd, MAX(days_past_due) AS max_dpd
FROM loans
GROUP BY status
""")

## Step 3 · Which loans are old enough to judge?

**What:** group loans by age at the snapshot date and compare default rates.
**Why:** a loan given out last month has not had time to default. Counting it as "good" makes the book look safer than it is (this is called *right-censoring*). So all risk figures below use only **mature loans**, at least 180 days old.

In [ ]:
sql("""
WITH aged AS (
    SELECT *, CAST(julianday('2025-12-31') - julianday(disbursement_date) AS INTEGER) AS age_days
    FROM loans
)
SELECT CASE WHEN age_days <= 180 THEN '1: up to 180 days'
            WHEN age_days <= 365 THEN '2: 181-365 days'
            ELSE '3: over 365 days' END              AS age_band,
       COUNT(*)                                       AS loans,
       SUM(status = 'Defaulted')                      AS defaults,
       ROUND(100.0 * SUM(status = 'Defaulted') / COUNT(*), 1) AS default_rate_pct
FROM aged
GROUP BY age_band
ORDER BY age_band
""")

Young loans show a lower default rate only because they have not had time to fail. Next we pull the mature cohort, joined to customer details. We also compute the **instalment-to-income ratio**: the monthly repayment as a share of monthly income, a simple affordability measure.

In [ ]:
MIN_AGE = 180
mature = sql(f"""
SELECT l.*, c.age, c.occupation, c.county, c.monthly_income_ksh,
       (l.status = 'Defaulted') AS is_default,
       l.principal_ksh * (1 + l.interest_rate_pct / 100.0 * l.term_months / 12)
           / l.term_months / c.monthly_income_ksh AS inst_to_income
FROM loans l
JOIN customers c ON c.customer_id = l.customer_id
WHERE julianday('2025-12-31') - julianday(l.disbursement_date) >= {MIN_AGE}
""")
all_loans = sql("SELECT COUNT(*) AS n, ROUND(100.0 * AVG(status = 'Defaulted'), 1) AS rate FROM loans").iloc[0]

print(f"All loans:    {int(all_loans['n'])} loans, default rate {all_loans['rate']}%")
print(f"Mature loans: {len(mature)} loans, default rate {100 * mature['is_default'].mean():.1f}%")

## Step 4 · Which segments default most?

**What:** default rate by product, credit-score band, instalment-to-income band and occupation.
**Why:** this shows *where* the risk sits. The `margin_pts` column is the 95% margin of error, so small groups are not over-read.

In [ ]:
def rate_table(d, col):
    g = d.groupby(col, observed=True).agg(loans=("loan_id", "count"), defaults=("is_default", "sum"))
    p = g["defaults"] / g["loans"]
    g["default_rate_pct"] = (100 * p).round(1)
    g["margin_pts"] = (196 * np.sqrt(p * (1 - p) / g["loans"])).round(1)
    return g

mature["score_band"] = pd.cut(mature["credit_score"], [0, 549, 599, 649, 699, 850],
                              labels=["<550", "550-599", "600-649", "650-699", "700+"])
mature["ratio_band"] = pd.cut(mature["inst_to_income"], [0, .10, .20, .35, .60, np.inf],
                              labels=["<10%", "10-20%", "20-35%", "35-60%", "60%+"])

by_product = rate_table(mature, "product").sort_values("default_rate_pct", ascending=False)
by_score = rate_table(mature, "score_band")
by_ratio = rate_table(mature, "ratio_band")
by_product

In [ ]:
by_ratio

In [ ]:
by_score

**Reading the tables.** Affordability is the clearest signal: once the instalment passes 60% of monthly income, more than half of loans default. Asset Finance has the highest default rate of the four products.

In [ ]:
fig, ax = plt.subplots(1, 2)
bp = by_product["default_rate_pct"]
ax[0].bar(bp.index, bp.values, color=[AMBER if v == bp.max() else BLUE for v in bp.values])
ax[0].set_title("Default rate by product"); ax[0].set_ylabel("Default rate (%)")
ax[0].tick_params(axis="x", labelrotation=20)
br = by_ratio["default_rate_pct"]
ax[1].bar(br.index.astype(str), br.values, color=[AMBER if v == br.max() else BLUE for v in br.values])
ax[1].set_title("Default rate by instalment-to-income band")
ax[1].set_xlabel("Instalment as share of monthly income"); ax[1].set_ylabel("Default rate (%)")
plt.tight_layout(); plt.show()

## Step 5 · Is risk getting better or worse? (vintages)

**What:** default rate by the month the loan was given out (its *vintage*), with a 3-month rolling average computed by a SQL **window function**.
**Why:** a single bad month can hide inside a yearly average. Vintages show it early.

In [ ]:
vint = sql(f"""
WITH v AS (
    SELECT strftime('%Y-%m', disbursement_date) AS vintage,
           COUNT(*)                              AS loans,
           SUM(status = 'Defaulted')             AS defaults,
           ROUND(100.0 * SUM(status = 'Defaulted') / COUNT(*), 1) AS default_rate_pct
    FROM loans
    WHERE julianday('2025-12-31') - julianday(disbursement_date) >= {MIN_AGE}
    GROUP BY vintage
)
SELECT *,
       ROUND(AVG(default_rate_pct) OVER (ORDER BY vintage
             ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS rolling3_pct
FROM v
ORDER BY vintage
""").set_index("vintage")
worst = vint["default_rate_pct"].idxmax()
print("Worst vintage:", worst, "at", vint.loc[worst, "default_rate_pct"], "% default")
vint.tail(6)

In [ ]:
fig, ax = plt.subplots()
ax.plot(vint.index, vint["default_rate_pct"], marker="o", color=BLUE, label="Vintage default rate")
ax.plot(vint.index, vint["rolling3_pct"], color=DARK, label="3-month rolling average")
ax.scatter([worst], [vint.loc[worst, "default_rate_pct"]], color=AMBER, s=70, zorder=3, label="Worst vintage")
ax.set_title("Default rate by disbursement month (mature loans)")
ax.set_xlabel("Disbursement month"); ax.set_ylabel("Default rate (%)")
ax.tick_params(axis="x", labelrotation=60); ax.legend()
plt.tight_layout(); plt.show()

The highest point, 2025-07, rests on only 2 loans, so it is a prompt to look closer rather than a trend. The rolling average gives the steadier picture.

## Step 6 · How much of the book is overdue? (PAR30 and PAR90)

**What:** *Portfolio at Risk* is the share of the outstanding balance on loans with a payment more than 30 (or 90) days late. Outstanding balance is estimated from the repayment schedule.
**Why:** PAR is the standard measure Kenyan lenders and regulators use to track loan book health.

In [ ]:
sql("""
WITH sched AS (
    SELECT loan_id,
           COUNT(*)                  AS n_due,
           SUM(paid_date IS NULL)    AS n_unpaid,
           MAX(CASE WHEN paid_date IS NULL
                    THEN julianday('2025-12-31') - julianday(due_date) ELSE 0 END) AS max_overdue_days
    FROM repayments
    GROUP BY loan_id
),
book AS (
    SELECT l.product,
           l.principal_ksh * (l.term_months - (s.n_due - s.n_unpaid)) * 1.0 / l.term_months AS outstanding,
           s.max_overdue_days
    FROM loans l JOIN sched s USING (loan_id)
)
SELECT product,
       COUNT(*)                          AS loans,
       ROUND(SUM(outstanding) / 1e6, 2)  AS outstanding_m,
       ROUND(100.0 * SUM(CASE WHEN max_overdue_days > 30 THEN outstanding ELSE 0 END) / SUM(outstanding), 1) AS par30_pct,
       ROUND(100.0 * SUM(CASE WHEN max_overdue_days > 90 THEN outstanding ELSE 0 END) / SUM(outstanding), 1) AS par90_pct
FROM book
WHERE outstanding > 0
GROUP BY product
ORDER BY par90_pct DESC
""")

**Note.** At the snapshot date no Digital Loans or Salary Advances are still active. The only balances left on those products belong to defaulted loans, so their PAR shows 100%. For live portfolio risk, read PAR on SME Loans and Asset Finance.

## Step 7 · What loss should we expect on active loans?

**What:** expected loss = **PD × LGD × EAD** for each product.
- **PD** (probability of default): the product's default rate on mature loans.
- **LGD** (loss given default): the share of the balance lost when a loan defaults. These are *assumptions*, stated below, because the data has no recovery records.
- **EAD** (exposure at default): the outstanding balance of active loans, from SQL.

**Why:** this is the number finance needs for provisioning.

In [ ]:
LGD = {"Digital Loan": 0.60, "Salary Advance": 0.50, "SME Loan": 0.45, "Asset Finance": 0.30}   # assumptions

ead = sql("""
SELECT l.product,
       COUNT(*) AS active_loans,
       SUM(l.principal_ksh * (l.term_months - (s.n_due - s.n_unpaid)) * 1.0 / l.term_months) AS ead_ksh
FROM loans l
JOIN (SELECT loan_id, COUNT(*) AS n_due, SUM(paid_date IS NULL) AS n_unpaid
      FROM repayments GROUP BY loan_id) s USING (loan_id)
WHERE l.status = 'Active'
GROUP BY l.product
""").set_index("product")

el = pd.DataFrame({"PD": mature.groupby("product")["is_default"].mean().round(3),
                   "LGD_assumed": pd.Series(LGD)}).join(ead).fillna(0)
el["active_loans"] = el["active_loans"].astype(int)
el["expected_loss_ksh"] = el["PD"] * el["LGD_assumed"] * el["ead_ksh"]
total_el = el["expected_loss_ksh"].sum()
print(f"Total expected loss on active loans: Ksh {total_el:,.0f} "
      f"({100 * total_el / el['ead_ksh'].sum():.1f}% of exposure)")
el.round({"ead_ksh": 0, "expected_loss_ksh": 0})

## Step 8 · Which approval rule makes the most money?

**What:** replay four approval rules on the mature loans. For each approved loan, a good loan earns its interest and a defaulted loan loses LGD × principal.
**Why:** a risk team should judge a rule by the shillings it makes, not only by how many defaults it blocks. Declining good customers costs money too.

In [ ]:
v = mature.copy()
v["interest_ksh"] = v["principal_ksh"] * v["interest_rate_pct"] / 100 * v["term_months"] / 12
v["value_ksh"] = np.where(v["is_default"] == 1, -v["product"].map(LGD) * v["principal_ksh"], v["interest_ksh"])

rules = {
    "Approve everyone":                     pd.Series(False, index=v.index),
    "Decline credit score under 550":       v["credit_score"] < 550,
    "Decline instalment over 60% of income": v["inst_to_income"] > 0.60,
    "Decline either":                       (v["credit_score"] < 550) | (v["inst_to_income"] > 0.60),
}
rows = []
for name, decline in rules.items():
    ok = ~decline
    rows.append([name, round(100 * ok.mean(), 1), round(100 * v.loc[ok, "is_default"].mean(), 1),
                 round(v.loc[ok, "value_ksh"].sum() / 1e6, 2)])
policy = pd.DataFrame(rows, columns=["rule", "approval_pct", "bad_rate_pct", "net_value_ksh_m"])
policy

## Key findings

In [ ]:
best = policy.loc[policy["net_value_ksh_m"].idxmax()]
base = policy.loc[policy["rule"] == "Approve everyone", "net_value_ksh_m"].iloc[0]
top = by_product["default_rate_pct"].idxmax()
print(f"- Mature-loan default rate: {100 * mature['is_default'].mean():.1f}% on {len(mature)} loans.")
print(f"- Instalment over 60% of income: {by_ratio.loc['60%+', 'default_rate_pct']}% default.")
print(f"- Highest-risk product: {top} ({by_product.loc[top, 'default_rate_pct']}%).")
print(f"- Worst vintage: {worst} ({vint.loc[worst, 'default_rate_pct']}%).")
print(f"- Expected loss on active loans: Ksh {total_el:,.0f}.")
print(f"- Best rule: '{best['rule']}', Ksh {best['net_value_ksh_m']}M vs Ksh {base}M approving everyone.")

## Recommendations

1. **Add an affordability cap:** decline instalments above 60% of income. It more than doubles net value while still approving about three in four applicants.
2. **Do not stack it with the score cut-off.** Declining on either rule loses more good business than it saves.
3. **Review Asset Finance terms first.** It defaults most and carries most of the expected loss.
4. **Report risk on mature loans only**, and track vintages monthly so a bad month is caught early.

**Limitations.** The data is synthetic. LGD values are assumptions. Rule values use full principal as the exposure, a simple and conservative choice.